In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.metrics import (roc_auc_score, precision_recall_curve, roc_curve, confusion_matrix, mean_squared_error, mean_absolute_error, r2_score)

In [ ]:
training_file_path = "/content/drive/MyDrive/Colab Notebooks/data-mining-cse572/flood-prediction-project/train.csv"

## MODEL DEVELOPMENT

#### CLASSIFICATION COMMON

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
RANDOM_STATE = 42

# Load
df = pd.read_csv(training_file_path)
FEATURES = [c for c in df.columns if c not in ["id","FloodProbability"]]
X = df[FEATURES].apply(lambda s: pd.to_numeric(s, errors="coerce")).astype("float32")
y = (df["FloodProbability"].astype("float32") >= 0.5).astype(int)

# Same-style split as your baseline (20% holdout)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

#### CLASSIFICATION WITH LOGISTIC REGRESSION

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

num_cols = list(X_tr.select_dtypes(include=[np.number]).columns)
pre = ColumnTransformer([("num", StandardScaler(with_mean=True), num_cols)], remainder="drop")

logreg = Pipeline([
    ("prep", pre),
    ("clf", LogisticRegression(
        solver="saga", penalty="l2",
        class_weight="balanced",
        C=1.0,
        max_iter=2000,
        tol=1e-3,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

logreg.fit(X_tr, y_tr)
proba = logreg.predict_proba(X_te)[:,1]
pred  = (proba >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_te, pred), "F1:", f1_score(y_te, pred))
print("Classification report:\n", classification_report(y_te, pred, digits=4))

Accuracy: 0.8473022290600737 F1: 0.8571548110152544
Classification report:
               precision    recall  f1-score   support

           0     0.8166    0.8563    0.8360    101616
           1     0.8752    0.8398    0.8572    121976

    accuracy                         0.8473    223592
   macro avg     0.8459    0.8481    0.8466    223592
weighted avg     0.8486    0.8473    0.8475    223592



#### CLASSIFICATION WITH RANDOM FOREST

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,      # a bit lighter than 500
    max_depth=None,        # let it grow
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced_subsample",
    bootstrap=True,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf.fit(X_tr, y_tr)
rf_pred = rf.predict(X_te)  # default 0.5 threshold internally
print("RF Accuracy:", accuracy_score(y_te, rf_pred), "F1:", f1_score(y_te, rf_pred))
print("RF Classification report:\n", classification_report(y_te, rf_pred, digits=4))

RF Accuracy: 0.8114869941679488 F1: 0.8268951185655499
RF Classification report:
               precision    recall  f1-score   support

           0     0.7913    0.7949    0.7931    101616
           1     0.8285    0.8253    0.8269    121976

    accuracy                         0.8115    223592
   macro avg     0.8099    0.8101    0.8100    223592
weighted avg     0.8116    0.8115    0.8115    223592



#### CLASSIFICATION WITH XGBOOST

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    tree_method="hist",
    n_estimators=500, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, reg_alpha=0.0,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    n_jobs=-1
)

xgb.fit(X_tr, y_tr)
xgb_pred = (xgb.predict_proba(X_te)[:,1] >= 0.5).astype(int)
print("XGB Accuracy:", accuracy_score(y_te, xgb_pred), "F1:", f1_score(y_te, xgb_pred))
print("XGB Classification report:\n", classification_report(y_te, xgb_pred, digits=4))

XGB Accuracy: 0.8362016530108412 F1: 0.8482460283917161
XGB Classification report:
               precision    recall  f1-score   support

           0     0.8118    0.8327    0.8221    101616
           1     0.8575    0.8392    0.8482    121976

    accuracy                         0.8362    223592
   macro avg     0.8347    0.8359    0.8352    223592
weighted avg     0.8367    0.8362    0.8364    223592



#### REGRESSION COMMON

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

RANDOM_STATE = 42

# --- Load & split ---
df = pd.read_csv(training_file_path)

FEATURES = [c for c in df.columns if c not in ["id","FloodProbability"]]
X = df[FEATURES].apply(lambda s: pd.to_numeric(s, errors="coerce")).astype("float32")
y = df["FloodProbability"].astype("float32")

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

num_cols = list(X_tr.select_dtypes(include=[np.number]).columns)
scaler = ColumnTransformer([("num", StandardScaler(with_mean=True), num_cols)],
                           remainder="drop")

def metrics(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{name:22s} | R2={r2:.4f}  RMSE={rmse:.5f}  MAE={mae:.5f}  "
          f"min/max={y_pred.min():.4f}/{y_pred.max():.4f}")

#### REGRESSION WITH LINEAR MODELS

In [ ]:
# Mean baseline
mean_pred = np.full_like(y_te, y_tr.mean())
metrics(y_te, mean_pred, "BaselineMean")

# Linear Regression
linreg = Pipeline([("prep", scaler), ("lr", LinearRegression())])
linreg.fit(X_tr, y_tr)
lin_pred = np.clip(linreg.predict(X_te), 0.0, 1.0)
metrics(y_te, lin_pred, "LinearRegression+clip")

# Ridge/Lasso
ridge = Pipeline([("prep", scaler), ("ridge", Ridge(alpha=1.0, random_state=RANDOM_STATE))])
ridge.fit(X_tr, y_tr); ridge_pred = np.clip(ridge.predict(X_te), 0.0, 1.0)
metrics(y_te, ridge_pred, "Ridge+clip")

lasso = Pipeline([("prep", scaler), ("lasso", Lasso(alpha=1e-4, random_state=RANDOM_STATE, max_iter=10000))])
lasso.fit(X_tr, y_tr); lasso_pred = np.clip(lasso.predict(X_te), 0.0, 1.0)
metrics(y_te, lasso_pred, "Lasso+clip")

BaselineMean           | R2=0.0000  RMSE=0.05098  MAE=0.04089  min/max=0.5045/0.5045
LinearRegression+clip  | R2=0.8449  RMSE=0.02008  MAE=0.01579  min/max=0.3025/0.7542
Ridge+clip             | R2=0.8449  RMSE=0.02008  MAE=0.01579  min/max=0.3025/0.7542
Lasso+clip             | R2=0.8448  RMSE=0.02009  MAE=0.01583  min/max=0.3047/0.7516


#### REGRESSION WITH GRADIENT BOOSTING AND RANDOM FOREST

In [ ]:
from sklearn.compose import TransformedTargetRegressor

def clip01(a, eps=1e-6):
    return np.clip(a, eps, 1-eps)

logit = FunctionTransformer(lambda z: np.log(clip01(z)/(1-clip01(z))),
                           inverse_func=lambda z: 1/(1+np.exp(-z)))

# (A) Gradient Boosting (fast, strong)
gbr = GradientBoostingRegressor(random_state=RANDOM_STATE)
gbr_bounded = TransformedTargetRegressor(regressor=gbr, transformer=logit)
gbr_bounded.fit(X_tr, y_tr)
gbr_pred = gbr_bounded.predict(X_te)  # already (0,1)
metrics(y_te, gbr_pred, "GBR_bounded")

# (B) Random Forest (robust, quick)
rf = RandomForestRegressor(
    n_estimators=500, max_depth=None, min_samples_leaf=1,
    max_features="sqrt", n_jobs=-1, random_state=RANDOM_STATE
)
rf_bounded = TransformedTargetRegressor(regressor=rf, transformer=logit)
rf_bounded.fit(X_tr, y_tr)
rf_pred = rf_bounded.predict(X_te)
metrics(y_te, rf_pred, "RF_bounded")

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_function_transformer.py:210: UserWarning: The provided functions are not strictly inverse of each other. If you are sure you want to proceed regardless, set 'check_inverse=False'.
  warnings.warn(


GBR_bounded            | R2=0.6175  RMSE=0.03153  MAE=0.02595  min/max=0.3909/0.6331


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_function_transformer.py:210: UserWarning: The provided functions are not strictly inverse of each other. If you are sure you want to proceed regardless, set 'check_inverse=False'.
  warnings.warn(


RF_bounded             | R2=0.6475  RMSE=0.03027  MAE=0.02484  min/max=0.3785/0.6433


#### REGRESSION WITH XGBOOST REGRESSOR

In [ ]:
from xgboost import XGBRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import FunctionTransformer
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score # Import necessary metrics

logit = FunctionTransformer(lambda y: np.log(y / (1 - y)),
                            inverse_func=lambda y_prime: 1 / (1 + np.exp(-y_prime)))

def metrics(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{name:22s} | R2={r2:.4f}  RMSE={rmse:.5f}  MAE={mae:.5f}  "
          f"min/max={y_pred.min():.4f}/{y_pred.max():.4f}")


xgb = XGBRegressor(
    tree_method="hist",
    n_estimators=600, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, reg_alpha=0.0,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_bounded = TransformedTargetRegressor(regressor=xgb, transformer=logit)
xgb_bounded.fit(X_tr, y_tr)
xgb_pred = xgb_bounded.predict(X_te)
metrics(y_te, xgb_pred, "XGB(gpu)_bounded")

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_function_transformer.py:210: UserWarning: The provided functions are not strictly inverse of each other. If you are sure you want to proceed regardless, set 'check_inverse=False'.
  warnings.warn(


XGB(gpu)_bounded       | R2=0.8298  RMSE=0.02103  MAE=0.01652  min/max=0.3246/0.7391
